This notebook will be used to implement Modality-Attentive Meta-Embedding Generator and integrate it into a single unified pipeline that handles warm-start and cold-start items without separate models.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
import json
import numpy as np
import torch
import pandas as pd

In [3]:
# Base paths (your standard structure)
base_dir = "/content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M"
preprocessed_dir = os.path.join(base_dir, "AntM2C_preprocessed")
fusion_artifacts_dir = os.path.join(base_dir, "fusion_artifacts")

# Key artifact paths
fusion_config_path = os.path.join(fusion_artifacts_dir, "fusion_config.json")

df_id_encoded_path = os.path.join(preprocessed_dir, "split_features", "df_id_encoded.pkl")

text_embeddings_path = os.path.join(preprocessed_dir, "text_embeddings.npy")
time_features_path = os.path.join(preprocessed_dir, "time_features_final.pt")

time_feature_cols_path = os.path.join(preprocessed_dir, "time_feature_columns.json")

image_embeddings_path = os.path.join(base_dir, "image_embeddings_vit_base_aligned.pt")
image_missing_mask_path = os.path.join(base_dir, "image_missing_mask.pt")

print("base_dir:", base_dir)
print("preprocessed_dir:", preprocessed_dir)
print("fusion_artifacts_dir:", fusion_artifacts_dir)

base_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M
preprocessed_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed
fusion_artifacts_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/fusion_artifacts


In [4]:
# Quick existence checks
paths_to_check = {
    "fusion_config.json": fusion_config_path,
    "df_id_encoded.pkl": df_id_encoded_path,
    "text_embeddings.npy": text_embeddings_path,
    "time_features_final.pt": time_features_path,
    "time_feature_columns.json": time_feature_cols_path,
    "image_embeddings_vit_base_aligned.pt": image_embeddings_path,
    "image_missing_mask.pt": image_missing_mask_path
}

for name, p in paths_to_check.items():
    print(f"{name} exists? -> {os.path.exists(p)} | {p}")

fusion_config.json exists? -> True | /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/fusion_artifacts/fusion_config.json
df_id_encoded.pkl exists? -> True | /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed/split_features/df_id_encoded.pkl
text_embeddings.npy exists? -> True | /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed/text_embeddings.npy
time_features_final.pt exists? -> True | /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed/time_features_final.pt
time_feature_columns.json exists? -> True | /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed/time_feature_columns.json
image_embeddings_vit_base_aligned.pt exists? -> True | /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/image_embeddings_vit_base_aligned.pt
image_missing_mask.pt exists? ->

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cpu


In [11]:
# Load fusion config + vocab sizes
with open(fusion_config_path, "r") as f:
    fusion_config = json.load(f)

print("--- Loaded fusion_config.json ---")
for k in ["num_users", "num_items", "num_scenes", "text_dim", "image_dim", "time_dim", "d_model", "num_experts", "hidden_dim"]:
    if k in fusion_config:
        print(f"{k}: {fusion_config[k]}")

--- Loaded fusion_config.json ---
num_users: 67615
num_items: 59142
num_scenes: 5
text_dim: 1536
image_dim: 768
time_dim: 5
d_model: 256
num_experts: 4
hidden_dim: 256


In [12]:
# Load df_id_encoded + validate columns
df_id = pd.read_pickle(df_id_encoded_path)
print("Loaded df_id_encoded.pkl:", df_id.shape)
print("Columns:", list(df_id.columns))

required_cols = ["user_id_idx", "item_id_idx"]
scene_candidates = ["scene_idx", "scene_id_idx"]

assert all(c in df_id.columns for c in required_cols), f"Missing required cols. Expected {required_cols}"
scene_col = next((c for c in scene_candidates if c in df_id.columns), None)
assert scene_col is not None, f"Missing scene column. Expected one of {scene_candidates}"

print("Detected scene column:", scene_col)

df_id.head()

Loaded df_id_encoded.pkl: (3730329, 3)
Columns: ['user_id_idx', 'item_id_idx', 'scene_idx']
Detected scene column: scene_idx


,user_id_idx,item_id_idx,scene_idx
0,41688,13590,4
1,39834,21351,2
2,22354,33398,2
3,17547,11190,1
4,55873,33398,2


In [17]:
# Load embeddings/features
# Text embeddings: memmap (interaction-level)
text_embeddings = np.load(text_embeddings_path, mmap_mode="r")
num_text_rows, text_dim = text_embeddings.shape

# Time features: torch tensor (interaction-level)
time_features = torch.load(time_features_path, map_location="cpu")
num_time_rows, time_dim = time_features.shape

# Time feature columns
with open(time_feature_cols_path, "r") as f:
    time_feature_cols = json.load(f)

# Image embeddings: aligned item-level
image_embeddings = torch.load(image_embeddings_path, map_location="cpu")
num_items_img, image_dim = image_embeddings.shape

# Missing mask
image_missing_mask = torch.load(image_missing_mask_path, map_location="cpu")
missing_count = int(image_missing_mask.sum().item())

In [18]:
# print shapes (lightweight)
print("------ Loaded Artifact Shapes ------")
print(f"df_id rows:        {len(df_id)}")
print(f"text_embeddings:   ({num_text_rows}, {text_dim})   [interaction-level]")
print(f"time_features:     ({num_time_rows}, {time_dim})   [interaction-level]")
print(f"image_embeddings:  ({num_items_img}, {image_dim})  [item-level aligned]")
print(f"missing images:    {missing_count}")
print("Time feature columns:", time_feature_cols)

------ Loaded Artifact Shapes ------
df_id rows:        3730329
text_embeddings:   (3730329, 1536)   [interaction-level]
time_features:     (3730329, 5)   [interaction-level]
image_embeddings:  (59142, 768)  [item-level aligned]
missing images:    2
Time feature columns: ['unix_time_norm', 'hour_sin', 'hour_cos', 'dayofweek', 'is_weekend']


In [20]:
# Alignment assert checks (should pass to proceed)
assert len(df_id) == num_text_rows, (
    f"Mismatch: df_id rows ({len(df_id)}) != text_embeddings rows ({num_text_rows})"
)
assert len(df_id) == num_time_rows, (
    f"Mismatch: df_id rows ({len(df_id)}) != time_features rows ({num_time_rows})"
)
assert len(time_feature_cols) == time_dim, (
    f"Mismatch: time_feature_cols ({len(time_feature_cols)}) != time_dim ({time_dim})"
)

assert num_items_img == fusion_config["num_items"], (
    f"Mismatch: image embedding rows ({num_items_img}) != num_items in fusion_config ({fusion_config['num_items']})"
)

print("Setup complete: all artifacts loaded and shapes are aligned.")

Setup complete: all artifacts loaded and shapes are aligned.


In [ ]:
%cd /content/drive/MyDrive/Final_Year_Project/Implementation